# Questions for all task objects

Generate one source-grounded Chinese question for each task in documents with nonzero workbook `task_count`. Profile-specific workbook rows determine eligibility; mirrored task records are counted once. Distinct corpus profiles remain distinct cases.

Run this notebook first, then `05.application.rag.task.ipynb`. Question authoring uses the configured RAG model at **172.27.74.16:11434**. It needs the saved `_tasks.json` documents but does not need a vector index. Source references remain reviewer metadata. The shared bank is published only after every selected task succeeds.

In [ ]:
# | notest
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

PROJ_ROOT = Path(os.getenv("PROJ_ROOT", Path.cwd())).resolve()
while not (PROJ_ROOT / "pyproject.toml").is_file() and PROJ_ROOT != PROJ_ROOT.parent:
    PROJ_ROOT = PROJ_ROOT.parent
if not (PROJ_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Set PROJ_ROOT to the ribosome project directory")
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))
load_dotenv(PROJ_ROOT / ".env", override=False)
from ribosome.ollama_config import OLLAMA_EMBED_HOST, OLLAMA_RAG_HOST, OLLAMA_TRUST_ENV

RAG_MODEL = os.getenv("OLLAMA_RAG_MODEL") or os.getenv("OLLAMA_SUMMARY_MODEL", "")
REQUEST_TIMEOUT = 300
BANK_PATH = PROJ_ROOT / "db/task_rag_tests/question_with_tasks/bank.json"
HEADERS = {"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]} if os.getenv("OLLAMA_API_KEY") else {}

from utils.task_rag_questions import (
    TaskQuestionAuthor, enumerate_task_cases, file_sha256, select_task_sources,
    validate_bank,
)
WORKBOOK = PROJ_ROOT / "proj_config/coding_categories/coding_manuals_markdown.all.xlsx"
CORPUS_DIRS = {profile: PROJ_ROOT / f"assets/code-docs.{profile}/.md" for profile in ("4.0", "5.0")}
AUTHORING_DIRECTORY = BANK_PATH.parent / "authored"
AUTHORING_CONCURRENCY = 2
PREVIEW_LIMIT = 20


## Select documents and enumerate tasks

The actual saved task objects determine coverage. Count differences are displayed for review; all tasks in each selected document are included. Missing, malformed, ambiguous, or empty task sources stop authoring with an explicit error.

In [ ]:
# | notest
selections = select_task_sources(PROJ_ROOT, WORKBOOK, CORPUS_DIRS)
task_cases, document_counts = enumerate_task_cases(PROJ_ROOT, selections)
display(pd.DataFrame(document_counts))
print(f"Selected {len(document_counts)} documents and {len(task_cases)} unique tasks")
if any(not row["count_matches"] for row in document_counts):
    print("Workbook/task count differences detected; every actual task will be included.")
selection_manifest = dict(
    workbook=str(WORKBOOK.relative_to(PROJ_ROOT)), workbook_sha256=file_sha256(WORKBOOK),
    documents=document_counts,
)


## Author and publish the complete bank

Only exact task content and source-derived context are supplied to the author. Stored summaries are excluded. Responses require one Chinese question and supporting quotations from the table. Successful checkpoints are reused; failed responses are retried on rerun. Each malformed response receives one corrective retry. A failure or interruption preserves the previous complete bank.

In [ ]:
# | notest
from ollama import AsyncClient
if not RAG_MODEL:
    raise ValueError("Configure OLLAMA_RAG_MODEL or OLLAMA_SUMMARY_MODEL")
authoring_client = AsyncClient(host=OLLAMA_RAG_HOST, headers=HEADERS,
                              timeout=REQUEST_TIMEOUT, trust_env=OLLAMA_TRUST_ENV)
try:
    author = TaskQuestionAuthor(authoring_client, RAG_MODEL, OLLAMA_RAG_HOST, AUTHORING_DIRECTORY)
    report = await author.publish(task_cases, document_counts, BANK_PATH,
                                  selection=selection_manifest, concurrency=AUTHORING_CONCURRENCY)
finally:
    await authoring_client._client.aclose()
display(pd.DataFrame(report["failures"]))
if not report["published"]:
    raise RuntimeError(f"{len(report['failures'])} task questions failed; previous bank preserved. Rerun this cell to retry.")
bank = validate_bank(report["bank"])
print(f"Published {len(bank['cases'])} questions: {BANK_PATH}")


## Cell 6 — six failures analysed and repaired

**Result: 6/6 failed cases repaired; the complete bank now contains 126/126 validated questions.** All 120 previously successful checkpoints were verified unchanged. The bank was published atomically at [bank.json](../db/task_rag_tests/question_with_tasks/bank.json). No RAG evaluation was run.

The error output above is the **historical output before this repair**. All six cases came from the 4.0 corpus, arc-welding manual B-5. Their original checkpoints recorded `ValueError: Provide one question rather than a question list`, but did not retain the rejected responses. Sequential diagnostic retries reproduced the multiple-question output for every case; those reproduced responses are quoted below.

**Shared fix:** keep the one-question validator strict, but make its corrective retry explicit: one Chinese question, one line, at most 1,000 characters, and only one final question mark; combine related steps into one task-level question. Keep exact quotation/source validation. Checkpoints now retain the case ID, each raw response, and its validation error, so future failures can be inspected directly. The prompt/model identity for successful checkpoints was preserved to allow cache reuse.

### 1. Intermittent welding: main program and subprogram — source task `164`

Case: `task_484db180be68275c5fc065d2` · section: **1 辅助焊接功能 → 1.12 断续焊 → 1.12.3 作业示例** · [source lines 3699–3796](../assets/code-docs.4.0/.md/06%20%E4%BA%A7%E5%93%81%E6%89%8B%E5%86%8C/%E8%BD%AF%E4%BB%B6%E7%B1%BB/%E6%89%8B%E5%86%8C/V5.0%E7%89%88%E6%9C%AC/SX322003%E3%80%8A%E6%96%B0%E6%9D%BE%E6%9C%BA%E5%99%A8%E4%BA%BA%E5%BC%A7%E7%84%8A%E5%BA%94%E7%94%A8%E6%93%8D%E4%BD%9C%E6%89%8B%E5%86%8C%E3%80%8B%28B-5%29/SX322003%E3%80%8A%E6%96%B0%E6%9D%BE%E6%9C%BA%E5%99%A8%E4%BA%BA%E5%BC%A7%E7%84%8A%E5%BA%94%E7%94%A8%E6%93%8D%E4%BD%9C%E6%89%8B%E5%86%8C%E3%80%8B%28B-5%29.md#L3699-L3796) · [retry checkpoint](../db/task_rag_tests/question_with_tasks/authored/447ad64226058d6e/7f96f5d0a73a007351f7ade6e3ddde5bdacd2cbb57ceb90c422421ce93829db2.json)

**Cause:** The response split main-program delegation and the subprogram counter loop into two independent questions. The reproduced response contained **2 question marks**.

**Reproduced rejected question:**

> 在断续焊作业示例中，主作业执行IMWON开启断续焊功能后，ARCL PR202指令为何不直接执行运动，而是必须通过CALL指令调用子作业来完成焊接动作？子作业中又是如何利用循环结构和计数器参数实现多段断续焊接的？

**Fix:** Ask one integrated question about how the main program and subprogram cooperate to execute intermittent welding.

**Validated replacement:**

> 在断续焊作业示例中，主作业通过IMWON开启断续焊功能后，ARCL PR202指令不执行运动而由CALL调用子作业完成焊接动作，子作业内部又通过L10循环、INC I2计数器以及ARCON起弧和ARCOFF熄弧指令实现多段焊接并最终到达B点，这一主作业与子作业协作的完整流程是如何实现断续焊焊接线的？

**Verification:** the corrective response passed the single-question, Chinese-language, source-address exclusion, and exact evidence-quotation checks. Status: **complete**.

### 2. Equal-distance spot welding — source task `166`

Case: `task_412c3227d5922a99567a1cce` · section: **1 辅助焊接功能 → 1.12 断续焊 → 1.12.4 等距点焊** · [source lines 3959–4020](../assets/code-docs.4.0/.md/06%20%E4%BA%A7%E5%93%81%E6%89%8B%E5%86%8C/%E8%BD%AF%E4%BB%B6%E7%B1%BB/%E6%89%8B%E5%86%8C/V5.0%E7%89%88%E6%9C%AC/SX322003%E3%80%8A%E6%96%B0%E6%9D%BE%E6%9C%BA%E5%99%A8%E4%BA%BA%E5%BC%A7%E7%84%8A%E5%BA%94%E7%94%A8%E6%93%8D%E4%BD%9C%E6%89%8B%E5%86%8C%E3%80%8B%28B-5%29/SX322003%E3%80%8A%E6%96%B0%E6%9D%BE%E6%9C%BA%E5%99%A8%E4%BA%BA%E5%BC%A7%E7%84%8A%E5%BA%94%E7%94%A8%E6%93%8D%E4%BD%9C%E6%89%8B%E5%86%8C%E3%80%8B%28B-5%29.md#L3959-L4020) · [retry checkpoint](../db/task_rag_tests/question_with_tasks/authored/447ad64226058d6e/932fa9b84a6078d916e6689ce57695eea620fb04aac0556783830c45d593ee06.json)

**Cause:** The response separately asked about the motion path, arc/delay sequence, and loop-count parameter, producing three questions. The reproduced response contained **3 question marks**.

**Reproduced rejected question:**

> 在等距点焊作业中，每次循环内焊枪从P3经P1到P4的运动路径是怎样的？起弧、点焊延时和熄弧的执行顺序是什么？循环次数由哪个参数决定？

**Fix:** Combine the path, welding sequence, and loop control into one question about the complete spot-welding cycle.

**Validated replacement:**

> 在等距点焊作业中，焊枪如何从P3经P1到P4完成一次点焊循环，起弧、点焊延时和熄弧按什么顺序执行，以及循环次数由哪个参数控制？

**Verification:** the corrective response passed the single-question, Chinese-language, source-address exclusion, and exact evidence-quotation checks. Status: **complete**.

### 3. Continuous spot welding — source task `171`

Case: `task_1ea71778e6826113671492d0` · section: **1 辅助焊接功能 → 1.13 连续点焊 → 1.13.3 作业示例** · [source lines 4172–4225](../assets/code-docs.4.0/.md/06%20%E4%BA%A7%E5%93%81%E6%89%8B%E5%86%8C/%E8%BD%AF%E4%BB%B6%E7%B1%BB/%E6%89%8B%E5%86%8C/V5.0%E7%89%88%E6%9C%AC/SX322003%E3%80%8A%E6%96%B0%E6%9D%BE%E6%9C%BA%E5%99%A8%E4%BA%BA%E5%BC%A7%E7%84%8A%E5%BA%94%E7%94%A8%E6%93%8D%E4%BD%9C%E6%89%8B%E5%86%8C%E3%80%8B%28B-5%29/SX322003%E3%80%8A%E6%96%B0%E6%9D%BE%E6%9C%BA%E5%99%A8%E4%BA%BA%E5%BC%A7%E7%84%8A%E5%BA%94%E7%94%A8%E6%93%8D%E4%BD%9C%E6%89%8B%E5%86%8C%E3%80%8B%28B-5%29.md#L4172-L4225) · [retry checkpoint](../db/task_rag_tests/question_with_tasks/authored/447ad64226058d6e/6925f79e063b1baba697c09d0848606ebb04bb025e5fff921c572745860b0716.json)

**Cause:** The response split the welding path and speed/acceleration settings into two questions. The reproduced response contained **2 question marks**.

**Reproduced rejected question:**

> 在连续点焊作业示例中，从起弧点经圆弧起点到焊缝终点的焊接路径是如何通过指令序列实现的？各关键位置点（起弧点、圆弧起点、焊缝终点）对应的速度和加速度参数分别是多少？

**Fix:** Ask one question about parameter configuration across the complete continuous spot-welding flow.

**Validated replacement:**

> 在连续点焊作业示例中，机器人从安全点出发，经起弧点进入连续焊焊接，再通过圆弧运动到达焊缝终点后熄弧并返回安全点，这一完整焊接流程中各阶段指令的速度与加速度参数是如何配置的？

**Verification:** the corrective response passed the single-question, Chinese-language, source-address exclusion, and exact evidence-quotation checks. Status: **complete**.

### 4. Combined one-dimensional and circular sensing — source task `286`

Case: `task_6e5af290bd46a5725fd3e3c8` · section: **1 智能焊接—电弧篇 → 1.7 传感组合 → 1.7.2 作业示例** · [source lines 6555–6704](../assets/code-docs.4.0/.md/06%20%E4%BA%A7%E5%93%81%E6%89%8B%E5%86%8C/%E8%BD%AF%E4%BB%B6%E7%B1%BB/%E6%89%8B%E5%86%8C/V5.0%E7%89%88%E6%9C%AC/SX322003%E3%80%8A%E6%96%B0%E6%9D%BE%E6%9C%BA%E5%99%A8%E4%BA%BA%E5%BC%A7%E7%84%8A%E5%BA%94%E7%94%A8%E6%93%8D%E4%BD%9C%E6%89%8B%E5%86%8C%E3%80%8B%28B-5%29/SX322003%E3%80%8A%E6%96%B0%E6%9D%BE%E6%9C%BA%E5%99%A8%E4%BA%BA%E5%BC%A7%E7%84%8A%E5%BA%94%E7%94%A8%E6%93%8D%E4%BD%9C%E6%89%8B%E5%86%8C%E3%80%8B%28B-5%29.md#L6555-L6704) · [retry checkpoint](../db/task_rag_tests/question_with_tasks/authored/447ad64226058d6e/65cc32cc885f328581569afa16cba1c6b18d465727addef6370f922080aac41a.json)

**Cause:** The response separately asked about the ARCL/ARCJ restriction and the sensing-point speed, producing two questions. The reproduced response contained **2 question marks**.

**Reproduced rejected question:**

> 在该传感组合作业中，三维位姿一维传感模块的传感接近点P3为何不能使用ARCJ而必须使用ARCL？该模块中传感点P4的运动速度VL被设置为多少？

**Fix:** Use one question connecting the documented motion restriction with the ordering of the sensing modules in the welding flow.

**Validated replacement:**

> 在智能焊接电弧篇的传感组合作业中，三维位姿一维传感模块的传感接近点为何必须使用ARCL而不能使用ARCJ，且该模块与圆弧传感模块的初始化及结束指令在整体焊接流程中是如何依次衔接的？

**Verification:** the corrective response passed the single-question, Chinese-language, source-address exclusion, and exact evidence-quotation checks. Status: **complete**.

### 5. Combined sensing with a multipass welding loop — source task `288`

Case: `task_131a7abac2f2da0e89c15932` · section: **1 智能焊接—电弧篇 → 1.7 传感组合 → 1.7.2 作业示例** · [source lines 6708–6917](../assets/code-docs.4.0/.md/06%20%E4%BA%A7%E5%93%81%E6%89%8B%E5%86%8C/%E8%BD%AF%E4%BB%B6%E7%B1%BB/%E6%89%8B%E5%86%8C/V5.0%E7%89%88%E6%9C%AC/SX322003%E3%80%8A%E6%96%B0%E6%9D%BE%E6%9C%BA%E5%99%A8%E4%BA%BA%E5%BC%A7%E7%84%8A%E5%BA%94%E7%94%A8%E6%93%8D%E4%BD%9C%E6%89%8B%E5%86%8C%E3%80%8B%28B-5%29/SX322003%E3%80%8A%E6%96%B0%E6%9D%BE%E6%9C%BA%E5%99%A8%E4%BA%BA%E5%BC%A7%E7%84%8A%E5%BA%94%E7%94%A8%E6%93%8D%E4%BD%9C%E6%89%8B%E5%86%8C%E3%80%8B%28B-5%29.md#L6708-L6917) · [retry checkpoint](../db/task_rag_tests/question_with_tasks/authored/447ad64226058d6e/962ca739c36eec89b8707dcdbf15b386b82d87e04dad31ce469bf23db74d8ad3.json)

**Cause:** The response split contact/start-point detection parameters and SEARCH calling differences into two questions. The reproduced response contained **2 question marks**.

**Reproduced rejected question:**

> 在该传感组合作业示例的多道焊循环（MLLP/MLCC ON 至 MLLP/MLCC OFF）中，第一条焊缝使用 ARCL P19 VL=5 进行直线焊接，第二条焊缝使用 ARCC P24 VC=20 和 ARCC P25 VC=20 进行圆弧焊接；两条焊缝在接触探测和起始点检测指令上分别使用了什么参数？第一条焊缝的 SEARCH 指令与第二条焊缝的 SEARCH TYPE=1 指令在调用方式上有何区别？

**Fix:** Focus one question on the documented motion-instruction and speed differences between the two weld paths.

**Validated replacement:**

> 在智能焊接电弧篇的传感组合作业示例中，多道焊循环区间内第一条焊缝通过 ARCL P19 VL=5 执行直线焊接，第二条焊缝通过 ARCC P24 VC=20 和 ARCC P25 VC=20 执行圆弧焊接，这两条焊缝在焊接路径指令类型和速度参数上存在怎样的差异？

**Verification:** the corrective response passed the single-question, Chinese-language, source-address exclusion, and exact evidence-quotation checks. Status: **complete**.

### 6. Laser tracking welding — source task `403`

Case: `task_4517717e2f68b0a2b776f4dd` · section: **2 智能焊接—激光篇 → 2.3 激光跟踪 → 2.3.1 激光跟踪 → 2.3.1.3 作业示例** · [source lines 8931–8988](../assets/code-docs.4.0/.md/06%20%E4%BA%A7%E5%93%81%E6%89%8B%E5%86%8C/%E8%BD%AF%E4%BB%B6%E7%B1%BB/%E6%89%8B%E5%86%8C/V5.0%E7%89%88%E6%9C%AC/SX322003%E3%80%8A%E6%96%B0%E6%9D%BE%E6%9C%BA%E5%99%A8%E4%BA%BA%E5%BC%A7%E7%84%8A%E5%BA%94%E7%94%A8%E6%93%8D%E4%BD%9C%E6%89%8B%E5%86%8C%E3%80%8B%28B-5%29/SX322003%E3%80%8A%E6%96%B0%E6%9D%BE%E6%9C%BA%E5%99%A8%E4%BA%BA%E5%BC%A7%E7%84%8A%E5%BA%94%E7%94%A8%E6%93%8D%E4%BD%9C%E6%89%8B%E5%86%8C%E3%80%8B%28B-5%29.md#L8931-L8988) · [retry checkpoint](../db/task_rag_tests/question_with_tasks/authored/447ad64226058d6e/5a3f48b4c34e71bba7e97b8ec9bc786ee694463a51b475467c6c12f6ae3e73f6.json)

**Cause:** The response independently asked about operation order, TRACKON settings, and arc-start/arc-end files, producing three questions. The reproduced response contained **3 question marks**.

**Reproduced rejected question:**

> 在激光跟踪焊接作业中，从寻位起点到焊接终点之间，机器人依次执行了哪些关键操作？其中TRACKON指令的跟踪文件号、Y方向偏移、Z方向偏移及补偿值分别设置为什么？起弧和收弧各引用了哪个文件？

**Fix:** Combine the tracking settings and referenced welding files into one question situated in the complete laser-tracking welding flow.

**Validated replacement:**

> 在激光跟踪焊接作业中，从寻位起点经起弧、焊接到收弧的完整流程里，TRACKON指令的跟踪文件号、Y方向偏移、Z方向偏移及补偿值分别设置为何值，起弧与收弧又各引用了哪个文件？

**Verification:** the corrective response passed the single-question, Chinese-language, source-address exclusion, and exact evidence-quotation checks. Status: **complete**.

### Rerunning cell 6

The six repaired checkpoints and the other 120 successful checkpoints are reusable; rerunning cell 6 will load them and skip question-generation requests when the inputs and authoring settings are unchanged. Restart the kernel and rerun setup to load the improved retry diagnostics. The historical failure output was retained for comparison.


## Bounded question preview

In [ ]:
# | notest
display(pd.DataFrame([
    dict(case_id=case["id"], category=case["category"], profile=case["profile"],
         task_id=case["task_id"], section=" → ".join(case["heading_path"]),
         source=case["source_md"], line_start=case["references"][0]["line_start"],
         line_end=case["references"][0]["line_end"], question=case["question"])
    for case in bank["cases"][:PREVIEW_LIMIT]
]))
print(f"Showing up to {PREVIEW_LIMIT} of {bank['task_count']} task questions")
